In [8]:
import os
REPO_URL = "https://github.com/minyi-k03/VideoProcessing-Project.git"
if os.path.isdir("/content/repo/.git"):
    !git -C /content/repo pull --ff-only          # 이미 받아 둔 경우 최신으로
else:
    !git clone --branch YongMin --depth 1 {REPO_URL} /content/repo
!git -C /content/repo log --oneline -5             # T1.3 커밋들(feat: 시드 하나에서 ... (T1.3))이 보여야 함. 안 보이면 로컬에서 git push 먼저
assert os.path.isdir("/content/repo/src/sr_pipeline"), "clone 실패: REPO_URL/푸시 확인"
print("셀 1 OK")


Already up to date.
cbb8b2f (HEAD -> YongMin, origin/YongMin) docs: T1.3 상태를 진행으로 되돌림 (Colab T4 CUDA 시드 확인 후 통과로 변경)
eda9a36 test: set_model_seed의 random·numpy 전역 시드와 cuDNN 결정적 모드 검증 추가, 픽스처가 CUDA RNG도 복원 (T1.3 Gemini 교차 리뷰 반영)
843550d docs: mark T1.3 as passed (code-review 반영: 시드 검증 명시, 테스트 전역 상태 복원)
95a84ba feat: 시드 하나에서 모델 초기화 난수와 데이터 난수를 분리 파생 (T1.3)
a222589 docs: mark T1.2 as passed (Gemini review applied, Colab CUDA/BasicSR check done, awaiting partner approval for main)
셀 1 OK


In [9]:
import sys, importlib, torch, numpy as np
from torch import nn
sys.path.insert(0, "/content/repo/src")
sys.path_importer_cache.pop("/content/repo/src", None)   # 클론 전에 import를 시도했던 커널의 "경로 없음" 캐시 제거
importlib.invalidate_caches()
# 이전 실행에서 import한 옛 모듈이 커널 메모리에 남아 있으면 git pull 후에도 옛 코드가 실행된다 -> 캐시를 비운다.
for name in [m for m in sys.modules if m == "sr_pipeline" or m.startswith("sr_pipeline.")]:
    del sys.modules[name]
from sr_pipeline.common.seed import set_model_seed, create_data_rng
# 디스크 파일이 아니라 실제로 로드된 함수의 코드를 검사한다(inspect.getsource는 디스크를 읽어서 옛 모듈을 못 잡는다).
loaded_consts = set_model_seed.__code__.co_consts + create_data_rng.__code__.co_consts
assert not any(isinstance(c, str) and c.startswith("TODO") for c in loaded_consts), \
    "seed.py가 구현 전 옛 버전입니다: 로컬에서 git push 한 뒤 셀 1을 다시 실행하세요."

assert torch.cuda.is_available(), "GPU 런타임이 아님"
dev = torch.device("cuda")
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))

def cuda_weights(seed):
    set_model_seed(seed)
    return nn.Conv2d(3, 8, 5, device=dev).weight.detach().clone()   # CUDA 난수로 초기화

def cuda_randn(seed):
    set_model_seed(seed)
    return torch.randn(64, device=dev)

def draw(rng):
    return rng.integers(0, 2**31 - 1, size=16)

results = {}
results["같은 시드 -> CUDA 가중치 초기화 동일"] = torch.equal(cuda_weights(1), cuda_weights(1))
results["다른 시드(0,1,2) -> CUDA 초기화 모두 다름"] = all(
    not torch.equal(cuda_weights(a), cuda_weights(b)) for a, b in [(0, 1), (0, 2), (1, 2)])
results["같은 시드 -> torch.randn(cuda) 동일"] = torch.equal(cuda_randn(2), cuda_randn(2))

# 데이터 수열은 모델·CUDA 난수 소비량과 무관해야 한다
reference = draw(create_data_rng(2))
set_model_seed(2)
nn.Conv2d(3, 64, 5, device=dev)
torch.randn(1000, device=dev)
results["CUDA 난수를 소비한 뒤에도 데이터 수열 불변"] = np.array_equal(draw(create_data_rng(2)), reference)

# 반대로 데이터 난수 사용이 CUDA 난수 수열을 바꾸면 안 된다
set_model_seed(0); a = torch.randn(8, device=dev)
set_model_seed(0); draw(create_data_rng(0)); b = torch.randn(8, device=dev)
results["데이터 RNG 사용이 CUDA 난수 수열을 안 바꿈"] = torch.equal(a, b)

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True
set_model_seed(0)
results["cuDNN deterministic=True, benchmark=False"] = (
    torch.backends.cudnn.deterministic is True and torch.backends.cudnn.benchmark is False)

for name, ok in results.items():
    print("OK  " if ok else "FAIL", name)
print("전체:", "PASS" if all(results.values()) else "FAIL")


torch 2.14.1+cu130 | Tesla T4
OK   같은 시드 -> CUDA 가중치 초기화 동일
OK   다른 시드(0,1,2) -> CUDA 초기화 모두 다름
OK   같은 시드 -> torch.randn(cuda) 동일
OK   CUDA 난수를 소비한 뒤에도 데이터 수열 불변
OK   데이터 RNG 사용이 CUDA 난수 수열을 안 바꿈
OK   cuDNN deterministic=True, benchmark=False
전체: PASS


In [10]:
# 참고(합격 기준 아님): 작은 ESPCN형 모델을 같은 시드로 2회 학습해 GPU에서 비트 단위로 같은지 본다.
# 결과는 T3.7(학습)에서 torch.use_deterministic_algorithms / CUBLAS_WORKSPACE_CONFIG가 필요한지 판단하는 근거다.
import torch.nn.functional as F

def tiny_train(seed, steps=30):
    set_model_seed(seed)
    net = nn.Sequential(
        nn.Conv2d(3, 32, 5, padding=2), nn.Tanh(),
        nn.Conv2d(32, 16, 3, padding=1), nn.Tanh(),
        nn.Conv2d(16, 3 * 16, 3, padding=1), nn.PixelShuffle(4)).to(dev)
    opt = torch.optim.Adam(net.parameters(), 1e-3)
    rng = create_data_rng(seed)
    for _ in range(steps):
        lr = torch.from_numpy(rng.random((8, 3, 24, 24), dtype=np.float32)).to(dev)
        hr = torch.from_numpy(rng.random((8, 3, 96, 96), dtype=np.float32)).to(dev)
        loss = F.l1_loss(net(lr), hr)
        opt.zero_grad(); loss.backward(); opt.step()
    return torch.cat([p.detach().flatten() for p in net.parameters()]).cpu()

w0a, w0b, w1 = tiny_train(0), tiny_train(0), tiny_train(1)
print("같은 시드 2회 -> 가중치 비트 단위 동일:", torch.equal(w0a, w0b), "| 최대 차이:", (w0a - w0b).abs().max().item())
print("다른 시드 -> 가중치 다름:", not torch.equal(w0a, w1))


같은 시드 2회 -> 가중치 비트 단위 동일: True | 최대 차이: 0.0
다른 시드 -> 가중치 다름: True
